> **Tip**: Welcome to the Investigate a Dataset project! You will find tips in quoted sections like this to help organize your approach to your investigation. Once you complete this project, remove these **Tip** sections from your report before submission. First things first, you might want to double-click this Markdown cell and change the title so that it reflects your dataset and investigation.

# Project: Investigate a Dataset - [Dataset-name]

## Table of Contents
<ul>
<li><a href="#intro">Introduction</a></li>
<li><a href="#wrangling">Data Wrangling</a></li>
<li><a href="#eda">Exploratory Data Analysis</a></li>
<li><a href="#conclusions">Conclusions</a></li>
</ul>

<a id='intro'></a>
## Introduction

### Dataset Description 

> **Tip**: In this section of the report, provide a brief introduction to the dataset you've selected/downloaded for analysis. Read through the description available on the `Project Details` page of `Investigate a Dataset` lesson for this course. List all column names in each table, and their significance. In case of multiple tables, describe the relationship between tables. 


### Question(s) for Analysis
>**Tip**: Clearly state one or more questions that you plan on exploring over the course of the report. You will address these questions in the **data analysis** and **conclusion** sections. Try to build your report around the analysis of at least one dependent variable and three independent variables. If you're not sure what questions to ask, then make sure you familiarize yourself with the dataset, its variables and the dataset context for ideas of what to explore.

> **Tip**: Once you start coding, use NumPy arrays, Pandas Series, and DataFrames where appropriate rather than Python lists and dictionaries. Also, **use good coding practices**, such as, define and use functions to avoid repetitive code. Use appropriate comments within the code cells, explanation in the mark-down cells, and meaningful variable names. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

<a id='wrangling'></a>
## Data Wrangling

### General Properties

- 110527 records found
- no empty fields found

In [3]:
df = pd.read_csv(
    "./Database_No_show_appointments/noshowappointments-kagglev2-may-2016.csv"
)

df = df.convert_dtypes()

print(f"rows: {df.shape[0]}")
print(f"any na?: {df.isna().sum().any()}")
print(f"any null?: {df.isnull().sum().any()}\n")

print(df.info())

rows: 110527
any na?: False
any null?: False

<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  Float64
 1   AppointmentID   110527 non-null  Int64  
 2   Gender          110527 non-null  string 
 3   ScheduledDay    110527 non-null  string 
 4   AppointmentDay  110527 non-null  string 
 5   Age             110527 non-null  Int64  
 6   Neighbourhood   110527 non-null  string 
 7   Scholarship     110527 non-null  Int64  
 8   Hipertension    110527 non-null  Int64  
 9   Diabetes        110527 non-null  Int64  
 10  Alcoholism      110527 non-null  Int64  
 11  Handcap         110527 non-null  Int64  
 12  SMS_received    110527 non-null  Int64  
 13  No-show         110527 non-null  string 
dtypes: Float64(1), Int64(8), string(5)
memory usage: 12.8 MB
None


### Data Cleaning

#### Column names

- `No-show` renamed to `NoShow` for Python compatibility
- `Hipertension` renamed to `Hypertension` (US English spelling)
- `AppointmentID` renamed to `AppointmentId` for consistency

In [4]:
df.rename(
    columns={
        "No-show": "NoShow",
        "Hipertension": "Hypertension",
        "AppointmentID": "AppointmentId",
    },
    inplace=True,
)

#### AppointmentId

- no duplicate `AppointmentId`s found

In [5]:
print(f"duplicate AppointmentIds: {df['AppointmentId'].duplicated().sum()}")

duplicate AppointmentIds: 0


#### Timestamps

- `ScheduledDay` and `AppointmentDay` converted to `datetime` datatype
- `ScheduledDay`: datetime of the actual appointment - [ref](https://www.kaggle.com/datasets/joniarroba/noshowappointments)
- `AppointmentDay`: datetime the appointment was created - [ref](https://www.kaggle.com/datasets/joniarroba/noshowappointments)
- timestamps are GMT

In [7]:
df["ScheduledDay"] = pd.to_datetime(df["ScheduledDay"])
df["AppointmentDay"] = pd.to_datetime(df["AppointmentDay"])

df[["ScheduledDay", "AppointmentDay"]].head()

,ScheduledDay,AppointmentDay
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00


#### Age
- Droped rows with invalid Age values `Age < 0`
- `Age == 0` assumed to be an infant
- Right-sized data type to cover realistic range of ages

In [8]:
print("Age range:")
print(f'{df["Age"].nunique()} unique (min:{df["Age"].min()}, max:{df["Age"].max()})')
df = df[df["Age"] >= 0]
print(f'{df["Age"].nunique()} unique (min:{df["Age"].min()}, max:{df["Age"].max()})')
df["Age"] = df["Age"].astype("Int8")

Age range:
104 unique (min:-1, max:115)
103 unique (min:0, max:115)


#### Neighbourhood

- 81 unique neighbourhoods
- converted to categorical

In [15]:
print(f"Neighbourhood count: {df['Neighbourhood'].nunique()}")
df["Neighbourhood"] = df["Neighbourhood"].astype("category")

Neighbourhood count: 81


#### Binary columns


##### Categoricals

- `Gender` converted to categorical datatype

In [ ]:
print(df["Gender"].unique())
df["Gender"] = df["Gender"].astype("category")

# TODO: create custom category "Male", "Female"

<StringArray>
['F', 'M']
Length: 2, dtype: string


##### Encoded Booleans

Converted binary columns (`[0,1]`, `["Yes","No"]`) to boolean
- `Scholarship`,
- `Hypertension`,
- `Diabetes`,
- `Alcoholism`,
- `SMS_received`,
- `NoShow`

In [17]:
binaryColumns = [
    "Scholarship",
    "Hypertension",
    "Diabetes",
    "Alcoholism",
    "SMS_received",
    "NoShow",
]

for colName in binaryColumns:
    print(f"{colName}:\n{df[colName].unique()}\n")

df["Scholarship"] = df["Scholarship"].astype("boolean")
df["Hypertension"] = df["Hypertension"].astype("boolean")
df["Diabetes"] = df["Diabetes"].astype("boolean")
df["Alcoholism"] = df["Alcoholism"].astype("boolean")
df["SMS_received"] = df["SMS_received"].astype("boolean")
df["NoShow"] = df["NoShow"].map({"Yes": True, "No": False}).astype("boolean")

Scholarship:
<IntegerArray>
[0, 1]
Length: 2, dtype: Int64

Hypertension:
<IntegerArray>
[1, 0]
Length: 2, dtype: Int64

Diabetes:
<IntegerArray>
[0, 1]
Length: 2, dtype: Int64

Alcoholism:
<IntegerArray>
[0, 1]
Length: 2, dtype: Int64

SMS_received:
<IntegerArray>
[0, 1]
Length: 2, dtype: Int64

NoShow:
<StringArray>
['No', 'Yes']
Length: 2, dtype: string



#### Handcap

- **Handcap** is a count of the patients handicaps. See this [clarification](https://www.kaggle.com/datasets/joniarroba/noshowappointments/discussion/29699#229356) by the dataset creator.
- Right-sized datatype to fit value range

In [18]:
print(df["Handcap"].unique())
df["Handcap"] = df["Handcap"].astype("Int8")

<IntegerArray>
[0, 1, 2, 3, 4]
Length: 5, dtype: Int64


#### Data Cleanup results

`PatientId` column is unchanged

In [19]:
print("cleaned data:")
print(df.info())

cleaned data:
<class 'pandas.DataFrame'>
Index: 110526 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype              
---  ------          --------------   -----              
 0   PatientId       110526 non-null  Float64            
 1   AppointmentId   110526 non-null  Int64              
 2   Gender          110526 non-null  category           
 3   ScheduledDay    110526 non-null  datetime64[us, UTC]
 4   AppointmentDay  110526 non-null  datetime64[us, UTC]
 5   Age             110526 non-null  Int8               
 6   Neighbourhood   110526 non-null  category           
 7   Scholarship     110526 non-null  boolean            
 8   Hypertension    110526 non-null  boolean            
 9   Diabetes        110526 non-null  boolean            
 10  Alcoholism      110526 non-null  boolean            
 11  Handcap         110526 non-null  Int8               
 12  SMS_received    110526 non-null  boolean            
 13  NoShow          

<a id='eda'></a>
## Exploratory Data Analysis

> **Tip**: Now that you've trimmed and cleaned your data, you're ready to move on to exploration. **Compute statistics** and **create visualizations** with the goal of addressing the research questions that you posed in the Introduction section. You should compute the relevant statistics throughout the analysis when an inference is made about the data. Note that at least two or more kinds of plots should be created as part of the exploration, and you must  compare and show trends in the varied visualizations. Remember to utilize the visualizations that the pandas library already has available.



> **Tip**: Investigate the stated question(s) from multiple angles. It is recommended that you be systematic with your approach. Look at one variable at a time, and then follow it up by looking at relationships between variables. You should explore at least three variables in relation to the primary question. This can be an exploratory relationship between three variables of interest, or looking at how two independent variables relate to a single dependent variable of interest. Lastly, you  should perform both single-variable (1d) and multiple-variable (2d) explorations.


### Research Question 1 (Replace this header name!)

In [ ]:
# Use this, and more code cells, to explore your data. Don't forget to add
#   Markdown cells to document your observations and findings.

### Research Question 2  (Replace this header name!)

In [ ]:
# Continue to explore the data to address your additional research
#   questions. Add more headers as needed if you have more questions to
#   investigate.

<a id='conclusions'></a>
## Conclusions

> **Tip**: Finally, summarize your findings and the results that have been performed in relation to the question(s) provided at the beginning of the analysis. Summarize the results accurately, and point out where additional research can be done or where additional information could be useful.

> **Tip**: Make sure that you are clear with regards to the limitations of your exploration. You should have at least 1 limitation explained clearly. 

> **Tip**: If you haven't done any statistical tests, do not imply any statistical conclusions. And make sure you avoid implying causation from correlation!

> **Tip**: Once you are satisfied with your work here, check over your report to make sure that it is satisfies all the areas of the rubric (found on the project submission page at the end of the lesson). You should also probably remove all of the "Tips" like this one so that the presentation is as polished as possible.

## Submitting your Project 

> **Tip**: Before you submit your project, you need to create a .html or .pdf version of this notebook in the workspace here. To do that, run the code cell below. If it worked correctly, you should see output that starts with `NbConvertApp] Converting notebook`, and you should see the generated .html file in the workspace directory (click on the orange Jupyter icon in the upper left).

> **Tip**: Alternatively, you can download this report as .html via the **File** > **Download as** submenu, and then manually upload it into the workspace directory by clicking on the orange Jupyter icon in the upper left, then using the Upload button.

> **Tip**: Once you've done this, you can submit your project by clicking on the "Submit Project" button in the lower right here. This will create and submit a zip file with this .ipynb doc and the .html or .pdf version you created. Congratulations!

In [ ]:
# Running this cell will execute a bash command to convert this notebook to an .html file
!python -m nbconvert --to html Investigate_a_Dataset.ipynb